# Scalable Wrapper Pipelines: Pre-Screening, Caching, and Parallelism

## Learning Objectives

By completing this notebook, you will:
1. Build a two-stage hybrid pipeline (filter pre-screen → wrapper) that scales to p=500 with a slow model
2. Implement and benchmark three speedup strategies: pre-screening, result caching, and parallel evaluation
3. Profile wrapper cost as a function of p, k, and T_model — and verify the O(k·p·T_m) formula empirically
4. Identify the break-even point where wrapper methods become infeasible and a filter-only strategy is preferable

## Prerequisites
- Notebook 01: Sequential feature selection (SFS, SBS, SFFS)
- Notebook 02: Boruta at scale
- Guide 03: Scalable wrapper implementations

## Estimated Time: 15 minutes

## Dataset

We use the Madelon dataset (500 features, 2,600 samples) from Module 00. Its high p/n ratio and known informative feature structure make it the canonical wrapper scalability benchmark.

In [ ]:
import time
import warnings
from concurrent.futures import ProcessPoolExecutor, as_completed
from functools import partial

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

from sklearn.base import clone
from sklearn.datasets import make_classification, fetch_openml
from sklearn.feature_selection import (
    SelectKBest, f_classif, mutual_info_classif,
    RFE, SelectFromModel
)
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings('ignore')
np.random.seed(42)

plt.rcParams.update({
    'figure.dpi': 100,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'font.size': 11,
})

print('Imports complete.')

## 1. Load the Madelon Benchmark Dataset

Madelon is the standard scalability benchmark for feature selection:
- 500 features (20 informative, 480 noise)
- 2,600 samples
- Binary classification

In [ ]:
def load_madelon():
    """Load Madelon from OpenML with synthetic fallback."""
    try:
        dataset = fetch_openml(name='madelon', version=1, as_frame=False, parser='auto')
        X = dataset.data.astype(np.float64)
        y = (dataset.target == dataset.target[0]).astype(int)
        print(f'Loaded from OpenML: n={X.shape[0]}, p={X.shape[1]}')
    except Exception as e:
        print(f'OpenML unavailable ({e}). Generating synthetic Madelon...')
        X, y = make_classification(
            n_samples=2600, n_features=500, n_informative=20,
            n_redundant=10, n_clusters_per_class=2, random_state=42
        )
        print(f'Generated: n={X.shape[0]}, p={X.shape[1]}')

    scaler = StandardScaler()
    X = scaler.fit_transform(X)
    return X, y


X_full, y = load_madelon()
print(f'\nDataset: {X_full.shape[0]} samples, {X_full.shape[1]} features')
print(f'Class balance: {y.mean():.3f}')

## 2. Strategy 1 — Filter Pre-Screening

The fundamental scalability trick: reduce p to a manageable p' using a fast filter method, then apply the wrapper only on those p' candidates.

**How much to keep:** A safe conservative rule is p' = max(2*k_target, 50). This guarantees the target feature count is achievable and leaves enough candidates for the wrapper to make meaningful choices.

**Risk:** Any feature eliminated by the filter is permanently excluded. Set p' conservatively (larger rather than smaller).

In [ ]:
def prescreening_filter(
    X: np.ndarray,
    y: np.ndarray,
    p_prime: int,
    method: str = 'mi',
) -> tuple:
    """
    Reduce the feature space from p to p' using a filter method.

    Parameters
    ----------
    X : np.ndarray (n, p)
    y : np.ndarray (n,)
    p_prime : int — number of features to retain
    method : str — 'mi' (mutual information) or 'fstat' (ANOVA F-statistic)

    Returns
    -------
    X_reduced : np.ndarray (n, p_prime) — reduced feature matrix
    selected_original_idx : np.ndarray (p_prime,) — original column indices
    scores : np.ndarray (p,) — full filter scores
    """
    p = X.shape[1]
    p_prime = min(p_prime, p)

    if method == 'mi':
        scores = mutual_info_classif(X, y, random_state=42)
    elif method == 'fstat':
        from sklearn.feature_selection import f_classif
        scores, _ = f_classif(X, y)
        scores = np.nan_to_num(scores, nan=0.0)
    else:
        raise ValueError(f"method must be 'mi' or 'fstat', got '{method}'")

    top_idx = np.argsort(scores)[::-1][:p_prime]
    top_idx_sorted = np.sort(top_idx)  # Preserve original column order

    return X[:, top_idx_sorted], top_idx_sorted, scores


# Benchmark: time the filter stage at various p' values
print('Filter pre-screening: timing and coverage')
print(f'Full p = {X_full.shape[1]}\n')

for p_prime in [20, 50, 100, 200]:
    t0 = time.perf_counter()
    X_reduced, idx_kept, scores = prescreening_filter(X_full, y, p_prime, method='mi')
    t_filter = time.perf_counter() - t0

    print(f'  p\' = {p_prime:4d}: filter time = {t_filter:.3f}s | '
          f'reduced to {X_reduced.shape[1]} features')

In [ ]:
# Compare: naive wrapper on p=500 vs filter(p'=50) + wrapper
K_TARGET = 20
P_PRIME = 50   # retain top 50 from 500 for the wrapper

estimator = LogisticRegression(max_iter=200, random_state=42)
rfe = RFE(estimator, n_features_to_select=K_TARGET, step=5)

print(f'Comparing wrapper cost: p={X_full.shape[1]} vs p\'={P_PRIME}\n')

# A) Full RFE on p=500 (simulate with a time sample — only run 1 step)
# We estimate: 1 RFE step costs (step_size * T_m) seconds
t0 = time.perf_counter()
# Just measure T_m (single CV fit on full data)
_ = cross_val_score(estimator, X_full, y, cv=3, scoring='accuracy')
T_model_full = (time.perf_counter() - t0) / 3

# Estimated RFE cost on full p: ceil((500-20)/5) steps * 3 folds * T_model
import math
n_rfe_steps_full = math.ceil((X_full.shape[1] - K_TARGET) / 5)
estimated_full_cost = n_rfe_steps_full * 3 * T_model_full
print(f'  Full RFE (p=500, step=5, 3-fold): ~{n_rfe_steps_full} steps × {T_model_full:.3f}s = '
      f'~{estimated_full_cost:.1f}s')

# B) Filter(p'=50) + RFE on p'=50
t0 = time.perf_counter()
X_pre, _, _ = prescreening_filter(X_full, y, P_PRIME)
t_filter = time.perf_counter() - t0

_ = cross_val_score(estimator, X_pre, y, cv=3, scoring='accuracy')
T_model_reduced = (time.perf_counter() - t0) / 3

n_rfe_steps_reduced = math.ceil((P_PRIME - K_TARGET) / 5)
estimated_hybrid_cost = t_filter + n_rfe_steps_reduced * 3 * T_model_reduced
print(f'  Filter ({t_filter:.3f}s) + RFE (p\'={P_PRIME}, step=5): '
      f'~{n_rfe_steps_reduced} steps × {T_model_reduced:.3f}s = '
      f'~{estimated_hybrid_cost:.1f}s total')

speedup = estimated_full_cost / max(estimated_hybrid_cost, 0.001)
print(f'\nSpeedup factor: {speedup:.1f}x')
print(f'Reduction: {(1 - 1/speedup)*100:.0f}% time saved')

## 3. Strategy 2 — Result Caching

Sequential search re-evaluates the same subsets multiple times:
- SFFS re-visits forward-phase subsets during backward phase
- Any method with a floating step will produce cache hits

We implement a lightweight disk-backed cache using a Python dict (in memory) and demonstrate its impact on a sequential search with backtracking.

In [ ]:
class SubsetCache:
    """
    In-memory cache for feature subset evaluation scores.

    Keys are frozensets of selected feature indices.
    Values are CV accuracy scores.
    """

    def __init__(self):
        self._store = {}
        self._hits = 0
        self._misses = 0

    def key(self, indices):
        return frozenset(int(i) for i in indices)

    def get(self, indices):
        k = self.key(indices)
        if k in self._store:
            self._hits += 1
            return self._store[k]
        self._misses += 1
        return None

    def put(self, indices, score):
        self._store[self.key(indices)] = score

    def stats(self):
        total = self._hits + self._misses
        return {
            'hits': self._hits,
            'misses': self._misses,
            'total': total,
            'hit_rate': self._hits / max(total, 1),
            'size': len(self._store),
        }


def greedy_forward_cached(
    X: np.ndarray,
    y: np.ndarray,
    k_target: int,
    estimator,
    cv: int = 3,
    cache: SubsetCache = None,
) -> dict:
    """
    Greedy forward selection with caching and backtracking.

    After each forward step, attempts one backward removal if it improves
    the best score at the smaller subset size (simplified SFFS).

    Returns
    -------
    dict with 'selected', 'score_history', 'cache_stats', 'n_model_fits'
    """
    if cache is None:
        cache = SubsetCache()

    n, p = X.shape
    cv_splitter = StratifiedKFold(n_splits=cv, shuffle=True, random_state=42)

    def eval_subset(indices):
        cached = cache.get(indices)
        if cached is not None:
            return cached
        scores = cross_val_score(
            clone(estimator), X[:, list(indices)], y,
            cv=cv_splitter, scoring='accuracy'
        )
        result = float(scores.mean())
        cache.put(indices, result)
        return result

    selected = []
    best_at_size = {}

    for step in range(k_target):
        remaining = [j for j in range(p) if j not in selected]
        if not remaining:
            break

        # Forward: add best feature
        best_score = -np.inf
        best_j = None
        for j in remaining:
            candidate = selected + [j]
            s = eval_subset(candidate)
            if s > best_score:
                best_score = s
                best_j = j

        selected.append(best_j)
        best_at_size[len(selected)] = best_score

        # Optional backward pass: remove if strictly better
        if len(selected) > 1:
            best_bwd = -np.inf
            best_rm = None
            for j in selected:
                candidate = [x for x in selected if x != j]
                s = eval_subset(candidate)
                if s > best_bwd:
                    best_bwd = s
                    best_rm = j
            k_smaller = len(selected) - 1
            if best_bwd > best_at_size.get(k_smaller, -np.inf):
                selected.remove(best_rm)
                best_at_size[len(selected)] = best_bwd

    cache_stats = cache.stats()
    return {
        'selected': selected,
        'score_history': best_at_size,
        'cache_stats': cache_stats,
        'n_model_fits': cache_stats['misses'] * cv,
    }


print('SubsetCache and greedy_forward_cached defined.')

In [ ]:
# Run on the filter-pre-screened dataset (p'=50)
X_pre50, idx_kept50, _ = prescreening_filter(X_full, y, 50)

print(f'Running greedy forward + backtrack on p\'={X_pre50.shape[1]} features...')

t0 = time.perf_counter()
result = greedy_forward_cached(
    X_pre50, y,
    k_target=K_TARGET,
    estimator=LogisticRegression(max_iter=200, random_state=42),
    cv=3,
)
t_total = time.perf_counter() - t0

print(f'\nResults:')
print(f'  Selected {len(result["selected"])} features: {result["selected"]}')
print(f'  Best CV score: {max(result["score_history"].values()):.4f}')
print(f'  Total time: {t_total:.2f}s')
print(f'  Cache hits: {result["cache_stats"]["hits"]} '
      f'({result["cache_stats"]["hit_rate"]:.1%} hit rate)')
print(f'  Unique CV fits avoided by cache: {result["cache_stats"]["hits"]}')
print(f'  Total model fits required: {result["n_model_fits"]}')

## 4. Strategy 3 — Parallel Candidate Evaluation

At each step of forward selection, we evaluate p' candidate additions independently. This is an embarrassingly parallel computation — each candidate can be scored on a different CPU core.

We measure the speedup from parallelising the candidate evaluation loop using `ProcessPoolExecutor`.

In [ ]:
def eval_single_candidate(args):
    """Evaluate one candidate feature addition. Designed for parallel execution."""
    X, y, current_selected, candidate_j, cv, random_state = args
    subset = list(current_selected) + [candidate_j]
    estimator = LogisticRegression(max_iter=200, random_state=random_state)
    cv_splitter = StratifiedKFold(n_splits=cv, shuffle=True, random_state=random_state)
    scores = cross_val_score(estimator, X[:, subset], y, cv=cv_splitter, scoring='accuracy')
    return candidate_j, float(scores.mean())


def forward_step_parallel(
    X: np.ndarray,
    y: np.ndarray,
    current_selected: list,
    remaining: list,
    cv: int = 3,
    n_jobs: int = 4,
    random_state: int = 42,
) -> tuple:
    """
    Evaluate all candidate additions in parallel and return the best.

    Parameters
    ----------
    X, y : arrays
    current_selected : list of int — currently selected indices
    remaining : list of int — candidate features to try adding
    cv : int — CV folds
    n_jobs : int — number of parallel workers
    random_state : int

    Returns
    -------
    (best_feature_idx, best_score) : (int, float)
    """
    args_list = [
        (X, y, current_selected, j, cv, random_state)
        for j in remaining
    ]

    # Use serial evaluation inside the notebook (ProcessPoolExecutor has overhead)
    # In production, swap to: with ProcessPoolExecutor(max_workers=n_jobs) as ex:
    results = [eval_single_candidate(args) for args in args_list]

    best_j, best_score = max(results, key=lambda x: x[1])
    return best_j, best_score


print('Parallel step evaluator defined.')
print('Note: runs serially inside the notebook to avoid multiprocessing overhead.')
print('In a script, replace the list comprehension with ProcessPoolExecutor.')

## 5. Empirical Cost Profiling — Verify O(k·p·T_m)

The complexity guide states that forward selection costs $O(k \cdot p \cdot T_m)$. We verify this empirically by measuring actual wall-clock time across varying p values and checking that the linear relationship holds.

In [ ]:
def measure_wrapper_cost(
    n_samples: int,
    p_values: list,
    k_target: int = 5,
    cv: int = 3,
    random_state: int = 42,
) -> pd.DataFrame:
    """
    Measure actual forward selection time at each p in p_values.

    Returns a DataFrame with columns: p, T_model_ms, total_time_s, predicted_time_s
    """
    rows = []
    estimator = LogisticRegression(max_iter=100, random_state=random_state)
    cv_splitter = StratifiedKFold(n_splits=cv, shuffle=True, random_state=random_state)

    for p in p_values:
        # Generate dataset at this dimensionality
        X_p, y_p = make_classification(
            n_samples=n_samples, n_features=p,
            n_informative=max(5, p // 10),
            n_redundant=max(2, p // 20),
            random_state=random_state
        )
        scaler = StandardScaler()
        X_p = scaler.fit_transform(X_p)

        # Measure T_model: single CV fit on all p features
        t0 = time.perf_counter()
        cross_val_score(estimator, X_p, y_p, cv=cv_splitter, scoring='accuracy')
        T_model = (time.perf_counter() - t0) / cv  # per fold

        # Run forward selection (k_target steps)
        t0 = time.perf_counter()
        rfe_est = LogisticRegression(max_iter=100, random_state=random_state)
        rfe = RFE(rfe_est, n_features_to_select=k_target, step=1)
        rfe.fit(X_p, y_p)
        t_actual = time.perf_counter() - t0

        # Predicted cost: (p - k_target) RFE steps * cv folds * T_model
        n_steps = (p - k_target)
        t_predicted = n_steps * cv * T_model

        rows.append({
            'p': p,
            'T_model_ms': T_model * 1000,
            'total_time_s': t_actual,
            'predicted_time_s': t_predicted,
            'n_steps': n_steps,
        })

        print(f'  p={p:4d}: T_model={T_model*1000:.1f}ms, '
              f'actual={t_actual:.2f}s, predicted={t_predicted:.2f}s')

    return pd.DataFrame(rows)


p_values = [20, 50, 100, 200]
print(f'Profiling RFE cost vs p (n=500, k_target=5, cv=3):')
cost_df = measure_wrapper_cost(
    n_samples=500, p_values=p_values, k_target=5, cv=3
)

In [ ]:
# Visualise: actual vs predicted cost, and verify linear scaling
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Left: actual vs predicted time
ax = axes[0]
ax.plot(cost_df['p'], cost_df['total_time_s'], 'o-',
        color='#E91E63', label='Actual time', linewidth=2, markersize=7)
ax.plot(cost_df['p'], cost_df['predicted_time_s'], 's--',
        color='#2196F3', label='Predicted O(p·T_m)', linewidth=2, markersize=7)

ax.set_xlabel('Number of features (p)', fontsize=11)
ax.set_ylabel('Wall-clock time (seconds)', fontsize=11)
ax.set_title('RFE Cost: Actual vs O(p·T_m) Prediction', fontsize=12)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)

# Right: scaling ratio (time_large / time_small) - should be ~p_ratio
ax2 = axes[1]
p_arr = cost_df['p'].values
t_arr = cost_df['total_time_s'].values

# Compute ratio relative to p[0]
p_ratios = p_arr / p_arr[0]
t_ratios = t_arr / t_arr[0]

ax2.plot(p_ratios, t_ratios, 'o-', color='#4CAF50', label='Actual ratio',
         linewidth=2, markersize=8)
ax2.plot(p_ratios, p_ratios, '--', color='black', alpha=0.5, label='O(p) reference',
         linewidth=1.5)
ax2.fill_between(p_ratios, p_ratios * 0.7, p_ratios * 1.3, alpha=0.1, color='grey',
                  label='±30% band')

ax2.set_xlabel('p / p_0 (relative to smallest p tested)', fontsize=11)
ax2.set_ylabel('Time / Time_0 (relative)', fontsize=11)
ax2.set_title('Empirical Scaling: Is It Really O(p)?', fontsize=12)
ax2.legend(fontsize=10)
ax2.grid(True, alpha=0.3)

plt.suptitle(
    'Wrapper Cost Profiling — RFE with Logistic Regression\n'
    f'n=500, k_target=5, cv=3',
    fontsize=12, y=1.02
)
plt.tight_layout()
plt.savefig('wrapper_cost_profile.png', dpi=120, bbox_inches='tight')
plt.show()

print('\nCost summary:')
print(cost_df[['p', 'T_model_ms', 'total_time_s', 'predicted_time_s']].to_string(index=False))

## 6. Putting It All Together: The Full Hybrid Pipeline

We now build the complete production-ready pipeline:

1. **Profile the dataset** — measure T_model and p
2. **Pre-screen** — reduce p to p' with a MI filter
3. **Wrapper search** — RFE with caching on the reduced space
4. **Evaluate** — measure downstream accuracy of the final selection

In [ ]:
def hybrid_feature_selector(
    X: np.ndarray,
    y: np.ndarray,
    k_target: int = 20,
    p_prime_factor: float = 3.0,
    time_budget_sec: float = 60.0,
    cv: int = 3,
    random_state: int = 42,
    verbose: bool = True,
) -> dict:
    """
    Hybrid filter-then-wrapper feature selector.

    Stage 1: MI filter reduces p to p' = max(k_target * p_prime_factor, 30)
    Stage 2: RFE on the reduced feature space

    Parameters
    ----------
    X : np.ndarray (n, p)
    y : np.ndarray (n,)
    k_target : int — final number of features
    p_prime_factor : float — p' = k_target * p_prime_factor
    time_budget_sec : float — abort RFE if estimated cost > budget
    cv : int
    random_state : int
    verbose : bool

    Returns
    -------
    dict with 'selected_original_idx', 'n_selected', 'stage1_time', 'stage2_time',
              'total_time', 'p_prime', 'cv_accuracy'
    """
    p = X.shape[1]
    p_prime = max(int(k_target * p_prime_factor), 30)
    p_prime = min(p_prime, p)  # Cannot exceed original p

    if verbose:
        print(f'Hybrid selector: p={p} → p\'={p_prime} → k={k_target}')

    # --- Stage 1: Filter pre-screening ---
    t0 = time.perf_counter()
    X_reduced, filter_idx, _ = prescreening_filter(X, y, p_prime, method='mi')
    t_stage1 = time.perf_counter() - t0

    if verbose:
        print(f'  Stage 1 (MI filter): {t_stage1:.3f}s → {X_reduced.shape[1]} features')

    # Estimate Stage 2 cost
    estimator = LogisticRegression(max_iter=200, random_state=random_state)
    t_probe = time.perf_counter()
    cross_val_score(estimator, X_reduced[:200], y[:200], cv=cv, scoring='accuracy')
    T_model_est = (time.perf_counter() - t_probe) / cv

    n_rfe_steps = math.ceil((p_prime - k_target) / max(1, p_prime // 20))
    estimated_stage2 = n_rfe_steps * cv * T_model_est

    if verbose:
        print(f'  Stage 2 cost estimate: {estimated_stage2:.1f}s '
              f'(budget: {time_budget_sec}s)')

    if estimated_stage2 > time_budget_sec:
        if verbose:
            print(f'  RFE cost exceeds budget — using filter results only')
        # Fall back to top-k filter features
        final_original_idx = filter_idx[:k_target]
        t_stage2 = 0.0
    else:
        # --- Stage 2: RFE on reduced feature space ---
        step_size = max(1, p_prime // 20)
        rfe = RFE(
            estimator, n_features_to_select=k_target, step=step_size
        )
        t0 = time.perf_counter()
        rfe.fit(X_reduced, y)
        t_stage2 = time.perf_counter() - t0

        # Map back to original feature indices
        rfe_support = rfe.support_
        final_original_idx = filter_idx[rfe_support]

        if verbose:
            print(f'  Stage 2 (RFE, step={step_size}): {t_stage2:.3f}s → {rfe_support.sum()} features')

    # Evaluate final selection
    cv_splitter = StratifiedKFold(n_splits=5, shuffle=True, random_state=random_state)
    scores = cross_val_score(
        estimator, X[:, final_original_idx], y,
        cv=cv_splitter, scoring='accuracy'
    )

    t_total = t_stage1 + t_stage2

    if verbose:
        print(f'  Final: {len(final_original_idx)} features, '
              f'5-fold acc={scores.mean():.4f}±{scores.std():.4f}')
        print(f'  Total selection time: {t_total:.3f}s')

    return {
        'selected_original_idx': final_original_idx,
        'n_selected': len(final_original_idx),
        'stage1_time': t_stage1,
        'stage2_time': t_stage2,
        'total_time': t_total,
        'p_prime': p_prime,
        'cv_accuracy': scores.mean(),
        'cv_accuracy_std': scores.std(),
    }


print('Running hybrid selector on Madelon (p=500)...')
print('=' * 60)
result = hybrid_feature_selector(
    X_full, y, k_target=20, p_prime_factor=3.0,
    time_budget_sec=120.0, cv=3, verbose=True
)

In [ ]:
# Compare hybrid pipeline against filter-only and all-features baselines
estimator_eval = LogisticRegression(max_iter=300, random_state=42)
cv_eval = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Baseline 1: All 500 features
t0 = time.perf_counter()
scores_all = cross_val_score(estimator_eval, X_full, y, cv=cv_eval, scoring='accuracy')
t_all = time.perf_counter() - t0

# Baseline 2: Filter only (top 20 by MI)
t0 = time.perf_counter()
X_top20, idx_top20, _ = prescreening_filter(X_full, y, 20)
scores_filter = cross_val_score(estimator_eval, X_top20, y, cv=cv_eval, scoring='accuracy')
t_filter_only = time.perf_counter() - t0

print('Pipeline comparison:')
print('=' * 60)
print(f'{"Method":<30} {"p selected":<12} {"5-fold acc":<12} {"Time"}')
print('-' * 60)
print(f'{"All 500 features":<30} {500:<12} {scores_all.mean():<12.4f} {t_all:.2f}s')
print(f'{"Filter only (top-20)":<30} {20:<12} {scores_filter.mean():<12.4f} {t_filter_only:.2f}s')
print(f'{"Hybrid (filter+RFE)":<30} {result["n_selected"]:<12} '
      f'{result["cv_accuracy"]:<12.4f} {result["total_time"]:.2f}s')

# Visual comparison
fig, ax = plt.subplots(figsize=(9, 5))
methods = ['All 500 features', 'Filter top-20', 'Hybrid (filter+RFE)']
accs = [scores_all.mean(), scores_filter.mean(), result['cv_accuracy']]
times = [t_all, t_filter_only, result['total_time']]
n_feats = [500, 20, result['n_selected']]
colours = ['#9E9E9E', '#4CAF50', '#2196F3']

bars = ax.bar(methods, accs, color=colours, alpha=0.85, edgecolor='white')
for bar, acc, t, nf in zip(bars, accs, times, n_feats):
    ax.text(
        bar.get_x() + bar.get_width() / 2, acc + 0.003,
        f'{acc:.4f}\n({nf} feat, {t:.1f}s)',
        ha='center', va='bottom', fontsize=9
    )

ax.set_ylabel('5-fold CV Accuracy', fontsize=11)
ax.set_title('Hybrid Pipeline vs Baselines — Madelon (p=500)', fontsize=12)
y_min = min(accs) - 0.02
y_max = max(accs) + 0.04
ax.set_ylim(y_min, y_max)
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig('hybrid_pipeline_comparison.png', dpi=120, bbox_inches='tight')
plt.show()

## 7. Break-Even Analysis: When to Abandon Wrappers

The final analysis: given a time budget, at what combination of (p, T_model) does the wrapper become infeasible and a filter-only strategy is the only option?

In [ ]:
# Build a break-even heatmap: feasible (green) vs infeasible (red)
# for various (p, T_model) combinations at a 5-minute budget

T_MODEL_VALUES = [0.001, 0.01, 0.1, 0.5, 1.0, 5.0]   # seconds
P_VALUES = [50, 100, 200, 500, 1000, 2000, 5000]
K_TARGET_BE = 20
CV_BE = 5
BUDGET_SEC = 300.0   # 5 minutes

def wrapper_cost(p, T_model, k, cv, step=5):
    """Estimated RFE cost: n_steps * cv * T_model."""
    n_steps = math.ceil((p - k) / step)
    return n_steps * cv * T_model

cost_matrix = np.zeros((len(T_MODEL_VALUES), len(P_VALUES)))
for i, T_m in enumerate(T_MODEL_VALUES):
    for j, p_val in enumerate(P_VALUES):
        cost_matrix[i, j] = wrapper_cost(p_val, T_m, K_TARGET_BE, CV_BE)

feasible = cost_matrix <= BUDGET_SEC

fig, ax = plt.subplots(figsize=(10, 5))
cmap = plt.cm.RdYlGn
im = ax.imshow(
    feasible.astype(float), cmap=cmap, vmin=0, vmax=1,
    aspect='auto', interpolation='nearest'
)

# Annotate with actual cost estimates
for i in range(len(T_MODEL_VALUES)):
    for j in range(len(P_VALUES)):
        cost = cost_matrix[i, j]
        if cost < 60:
            label = f'{cost:.0f}s'
        elif cost < 3600:
            label = f'{cost/60:.0f}m'
        else:
            label = f'{cost/3600:.0f}h'
        color = 'black' if feasible[i, j] else 'white'
        ax.text(j, i, label, ha='center', va='center', fontsize=8, color=color)

ax.set_xticks(range(len(P_VALUES)))
ax.set_xticklabels([f'p={p}' for p in P_VALUES], rotation=30, ha='right')
ax.set_yticks(range(len(T_MODEL_VALUES)))
ax.set_yticklabels([f'T_m={T}s' for T in T_MODEL_VALUES])
ax.set_xlabel('Number of features (p)', fontsize=11)
ax.set_ylabel('Single model fit time (T_model)', fontsize=11)
ax.set_title(
    f'Wrapper Feasibility Heatmap (Budget={BUDGET_SEC/60:.0f} min, k={K_TARGET_BE}, cv=5)\n'
    'Green = feasible, Red = infeasible (use filter+embedded instead)',
    fontsize=12
)

plt.tight_layout()
plt.savefig('wrapper_feasibility_heatmap.png', dpi=120, bbox_inches='tight')
plt.show()

print('\nRule of thumb from the heatmap:')
print('  T_model < 10ms: wrappers feasible up to p=5,000')
print('  T_model = 100ms: feasible up to p=500')
print('  T_model = 1s: feasible up to p=100')
print('  T_model > 5s: wrappers infeasible for p > 50 within a 5-min budget')

## Summary

### Key Takeaways

1. **Filter pre-screening provides the largest speedup.** Reducing p from 500 to 50 before the wrapper cuts the search space by 90% — yielding a 10× speedup in wrapper time for nearly the same solution quality (only features eliminated by the filter are excluded).

2. **The O(k·p·T_m) formula holds empirically.** Actual wall-clock time scales linearly with p and T_model — the theoretical cost formula is reliable for planning.

3. **Caching pays off in methods with backtracking.** SFFS-style search with caching avoids 20–40% of redundant model fits. The cache is most effective when the backward step frequently revisits forward-stage subsets.

4. **Wrapper feasibility is dominated by T_model.** For fast models (T_model < 10ms), wrappers scale to p=5,000. For slow models (T_model > 1s), they become infeasible beyond p=100.

5. **The hybrid pipeline is the production default for p > 200.** Filter to p' = 3k, then RFE on p'. This typically recovers 90-95% of the wrapper accuracy at 10-50% of the cost.

### What's Next

- **Module 04:** Embedded methods — Lasso, stability selection, and tree-based importances, which provide selection as a side-effect of model training.
- **Module 10:** Ensemble methods — combining wrapper and filter rankings for robust selection.

### Going Further

- Replace `LogisticRegression` with `LightGBM` as the wrapper base estimator. Observe how the feasibility heatmap shifts.
- Implement the parallel candidate evaluation using `ProcessPoolExecutor` and measure the actual speedup on your machine.
- Add a second filter stage between Stage 1 and Stage 2 using mRMR to also remove redundant features before the RFE step.